In [10]:
using Pkg
Pkg.activate(@__DIR__)


  Activating project at `c:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\benchmark`


In [11]:
# Auto-develop the parent package if it is not yet in the environment
if !haskey(Pkg.project().dependencies, "TrustRegionRadius")
    @info "Developing TrustRegionRadius from parent directory…"
    Pkg.develop(PackageSpec(path = joinpath(@__DIR__, "..")))
end

In [12]:
Pkg.instantiate()

Precompiling project...
   7897.8 ms  ✓ TrustRegionRadius
  1 dependency successfully precompiled in 14 seconds. 228 already precompiled.


In [13]:
using Revise
using TrustRegionRadius
using CUTEst
using JLD2
using LinearAlgebra
using Printf

In [14]:
const SOLVER_PARAMS = TRSolverParams(
    η₁ = 0.1,
    η₂ = 0.9,
    Δ₀ = 1.0,
    max_iterations = 10_000,
    tol = 1e-5,
)

TRSolverParams:
  η₁: 0.1  η₂: 0.9
  Δ₀: 1.0
  max_iterations: 10000
  tol: 1.0e-5


In [15]:
# Factory functions so each run gets a fresh (and for R4 mutable) rule
const RULES = [
    ("R1", () -> R1ClassicalUpdate(0.25, 0.50, 2.0)),
    ("R2", () -> R2StepSizeUpdate(0.25, 0.80, 2.0)),
    ("R3", () -> R3DFOLikeUpdate(0.25, 0.50, 2.0, 1.0)),
    ("R4", () -> R4RelativeGradUpdate(0.25, 2.0,  1.0)),
]

4-element Vector{Tuple{String, Function}}:
 ("R1", var"#15#19"())
 ("R2", var"#16#20"())
 ("R3", var"#17#21"())
 ("R4", var"#18#22"())

In [16]:
try
    finalize(nlp) 
catch e
    @error "No model to finalize: $e"
end

┌ Error: No model to finalize: UndefVarError(:nlp, Main)
└ @ Main c:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\benchmark\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl:4


In [17]:
# =============================================================================
# Problem selection
# =============================================================================

@info "Querying CUTEst problem list…"

prob_name = "ROSENBR"
nlp = CUTEstModel(prob_name)

┌ Info: Querying CUTEst problem list…
└ @ Main c:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\benchmark\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W2sZmlsZQ==.jl:5


  Problem name: ROSENBR
   All variables: ████████████████████ 2      All constraints: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
            free: ████████████████████ 2                 free: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           lower: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                lower: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           upper: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                upper: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
         low/upp: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0              low/upp: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           fixed: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                fixed: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
          infeas: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0               infeas: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
            nnzh: (  0.00% sparsity)   3               linear: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
                                                    nonlinear: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
                                                         nnzj: (------% sparsity)         
                                                     lin_nnzj: (--

In [18]:
# Safety check: skip if problem has constraints or wrong size
@show nvar = nlp.meta.nvar
@show ncon = nlp.meta.ncon
if nvar < 2 || nvar > 500 || ncon > 0
    @warn "  $rule_name: skipping $prob_name (nvar=$nvar, ncon=$ncon)"
    finalize(nlp)
    nlp = nothing
end

nvar = nlp.meta.nvar = 2
ncon = nlp.meta.ncon = 0


In [33]:
rule_number = 3
rule = RULES[rule_number][2]()

R3 DFO-Like Update:
  γ₁ (contraction):   0.25
  γ₂ (no-expand):     0.5
  γ₃ (expansion):     2.0
  ζ  (threshold):     1.0


In [34]:
SOLVER_PARAMS

TRSolverParams:
  η₁: 0.1  η₂: 0.9
  Δ₀: 1.0
  max_iterations: 10000
  tol: 1.0e-5


In [35]:
t0  = time()
out = trust_region_solver(nlp, rule, SOLVER_PARAMS)
elapsed = time() - t0

0.4069998264312744

In [36]:
out

--------- TROutput ---------
  Status:             solved
  Iterations:         36
  f evaluations:      113
  g evaluations:      96
  h evaluations:      0
  h_prod evaluations: 298
  Final ‖g‖:          1.315656424949353e-6
  Final Δ:            0.03125
  Solve time (s):     0.0009999275207519531


In [37]:
out.delta_trajectory

37-element Vector{Float64}:
 1.0
 2.0
 4.0
 1.0
 0.25
 0.5
 1.0
 2.0
 0.5
 1.0
 ⋮
 0.25
 0.0625
 0.125
 0.0625
 0.125
 0.0625
 0.125
 0.0625
 0.03125